
# Unit 2 — Advanced Visualization & Storytelling (UE24CS342AA9)
## Activity Notebook: Build an Interactive Retail Sales Dashboard

### Scenario

You have just joined the analytics team of **UrbanCart**, a mid-sized retail chain with stores across
three regions. Regional managers currently receive a static monthly PDF report and complain that:

1. They can't drill into *why* a number moved without emailing the analytics team.
2. Comparing stores or products side by side means flipping between many separate PDF pages.
3. Spotting a sudden dip in a specific product/store is slow and easy to miss.

**Your job:** apply the interaction techniques from the lecture — dynamic queries, coordinated views
& brushing, table lens, focus+context drill-down, and single-screen dashboard design — to build an
**interactive exploration tool** that solves these three complaints.



## Task 0 — Load the Sample Dataset (provided)

Run the cells below as-is. This generates `sales_df`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from ipywidgets import interact, HBox, VBox
sns.set_theme(style="whitegrid")
np.random.seed(7)


In [ ]:
stores = pd.DataFrame({
    "store_id": ["S01", "S02", "S03", "S04", "S05", "S06"],
    "store_city": ["Bengaluru", "Mysuru", "Mumbai", "Pune", "Delhi", "Jaipur"],
    "region": ["South", "South", "West", "West", "North", "North"]
})
products = pd.DataFrame({
    "product": ["Wireless Earbuds", "Smartwatch", "Laptop Sleeve", "USB-C Hub", "Yoga Mat", "Dumbbell Set", "Running Shoes", "Track Jacket", "Air Fryer", "Blender"],
    "category": ["Electronics"] * 4 + ["Fitness"] * 2 + ["Apparel"] * 2 + ["Home"] * 2,
    "unit_price": [1499, 4999, 899, 1299, 799, 2499, 3499, 1999, 3999, 2299],
})
months = pd.date_range("2023-01-01", periods=24, freq="MS")
rows = []
for _, s in stores.iterrows():
    store_scale = np.random.uniform(0.7, 1.4)
    for _, p in products.iterrows():
        product_scale = np.random.uniform(0.6, 1.6)
        trend = np.random.normal(0.01, 0.01)
        for i, m in enumerate(months):
            seasonal = 1 + 0.25 * np.sin(2 * np.pi * (m.month / 12))
            units = max(0, np.random.poisson(15 * store_scale * product_scale * seasonal * (1 + trend) ** i))
            revenue = units * p["unit_price"]
            profit_margin = np.random.uniform(0.12, 0.35)
            profit = revenue * profit_margin
            rating = np.clip(np.random.normal(4.1, 0.4), 1, 5)
            rows.append((m, s["store_id"], s["store_city"], s["region"], p["product"], p["category"], p["unit_price"], units, revenue, profit, round(rating, 1)))
sales_df = pd.DataFrame(rows, columns=["month", "store_id", "store_city", "region", "product", "category", "unit_price", "units_sold", "revenue", "profit", "avg_rating"])
for _col in ["units_sold", "revenue", "profit"]:
    sales_df[_col] = sales_df[_col].astype(float)
mask = (sales_df.store_id == "S03") & (sales_df["product"] == "Smartwatch") & (sales_df.month == "2024-06-01")
sales_df.loc[mask, ["units_sold", "revenue", "profit"]] *= 0.15
print(sales_df.shape)
sales_df.head()


In [ ]:
revenue_by_product = sales_df.groupby("product")["revenue"].sum().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x=revenue_by_product.values, y=revenue_by_product.index, color="steelblue", ax=ax)
ax.set_xlabel("Total Revenue ($)")
ax.set_ylabel("Product")
ax.set_title("STATIC: Total Revenue by Product (sorted descending)")
plt.tight_layout()
plt.show()


In [ ]:
rev_cat = sales_df.groupby(["product", "category"], as_index=False)["revenue"].sum().sort_values("revenue", ascending=False)
fig = px.bar(rev_cat, x="product", y="revenue", color="category", hover_data={"revenue": ":,.0f", "category": True}, title="INTERACTIVE: Total Revenue by Product")
fig.update_layout(xaxis_tickangle=45, height=480)
fig.show()


In [ ]:
price_slider = widgets.FloatRangeSlider(value=[sales_df.unit_price.min(), sales_df.unit_price.max()], min=sales_df.unit_price.min(), max=sales_df.unit_price.max(), step=50, description="Price range:", continuous_update=True, layout=widgets.Layout(width="500px"))
units_slider = widgets.FloatRangeSlider(value=[sales_df.units_sold.min(), sales_df.units_sold.max()], min=sales_df.units_sold.min(), max=sales_df.units_sold.max(), step=1, description="Units range:", continuous_update=True, layout=widgets.Layout(width="500px"))
def dynamic_filter(price_range, units_range):
    filtered = sales_df[sales_df.unit_price.between(*price_range) & sales_df.units_sold.between(*units_range)]
    print(f"{len(filtered)} / {len(sales_df)} rows match")
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.scatter(sales_df.unit_price, sales_df.units_sold, s=10, c="lightgray")
    ax.scatter(filtered.unit_price, filtered.units_sold, s=25, c="crimson")
    ax.set_xlabel("Unit Price"); ax.set_ylabel("Units Sold")
    ax.set_title(f"Dynamic Query: {len(filtered)}/{len(sales_df)} rows match")
    plt.tight_layout(); plt.show()
interact(dynamic_filter, price_range=price_slider, units_range=units_slider)


In [ ]:
store_overview = sales_df.groupby("store_id").revenue.sum().reset_index()
def store_coordinated_view(selected_store):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    colors = ["crimson" if s == selected_store else "lightgray" for s in store_overview.store_id]
    axes[0].bar(store_overview.store_id, store_overview.revenue, color=colors)
    axes[0].set_title(f"Overview: Revenue by Store ({selected_store})")
    axes[0].set_ylabel("Total Revenue ($)")
    cat_rev = sales_df[sales_df.store_id == selected_store].groupby("category").revenue.sum()
    axes[1].pie(cat_rev.values, labels=cat_rev.index, autopct="%1.0f%%", startangle=90)
    axes[1].set_title(f"Category Breakdown: {selected_store}")
    trend = sales_df[sales_df.store_id == selected_store].groupby("month").revenue.sum()
    axes[2].plot(trend.index, trend.values, marker="o", color="crimson")
    axes[2].set_title(f"Monthly Revenue Trend: {selected_store}")
    axes[2].tick_params(axis="x", rotation=45)
    plt.tight_layout(); plt.show()
interact(store_coordinated_view, selected_store=widgets.Dropdown(options=list(stores["store_id"]), description="Store:"))


In [ ]:
def min_median_ratio(s):
    med = s.median()
    return s.min() / med if med != 0 else np.nan
anomaly_ranking = sales_df.groupby(["store_id", "product"]).units_sold.agg(min_median_ratio).sort_values().head(5)
print(anomaly_ranking)


In [ ]:
overview_line = px.line(sales_df.sort_values("month"), x="month", y="units_sold", color="store_id", line_group="product", hover_data=["store_id", "product"], title="OVERVIEW: units_sold trend per (store_id, product)")
overview_line.update_traces(line=dict(width=1), opacity=0.4)
overview_line.update_layout(height=450, showlegend=False)
overview_line.show()
top_store, top_product = anomaly_ranking.index[0]
focus = sales_df[(sales_df.store_id == top_store) & (sales_df.product == top_product)].sort_values("month")
context = sales_df[sales_df["product"] == top_product].groupby("month").units_sold.mean()
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(context.index, context.values, color="gray", alpha=0.6, label=f"Context: avg '{top_product}' trend, all stores")
ax.plot(focus.month, focus.units_sold, color="crimson", marker="o", label=f"Focus: {top_store} / {top_product}")
ax.set_title(f"Focus + Context Drill-Down: {top_store} / {top_product}")
ax.set_xlabel("Month"); ax.set_ylabel("Units Sold"); ax.legend(); plt.tight_layout(); plt.show()


In [ ]:
product_summary = sales_df.groupby("product", as_index=False).agg(revenue=("revenue", "sum"), profit=("profit", "sum"), avg_rating=("avg_rating", "mean")).sort_values("revenue", ascending=False).reset_index(drop=True)
styled = (product_summary.style.bar(subset=["revenue"], color="#5DADE2").bar(subset=["profit"], color="#58D68D").bar(subset=["avg_rating"], color="#F5B041").format({"revenue": ":,.0f", "profit": ":,.0f", "avg_rating": "{:.2f}"}).set_caption("Table Lens: product performance, sorted by revenue"))
styled


In [ ]:
product_summary_by_profit = product_summary.sort_values("profit", ascending=False).reset_index(drop=True)
styled_profit = (product_summary_by_profit.style.bar(subset=["revenue"], color="#5DADE2").bar(subset=["profit"], color="#58D68D").bar(subset=["avg_rating"], color="#F5B041").format({"revenue": ":,.0f", "profit": ":,.0f", "avg_rating": "{:.2f}"}).set_caption("Table Lens: same products, re-sorted by profit"))
styled_profit


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0, 0].axis("off")
kpi_text = f"Total revenue: ${sales_df.revenue.sum():,.0f}\nTotal profit: ${sales_df.profit.sum():,.0f}\nAvg rating: {sales_df.avg_rating.mean():.2f}\nActive stores: {sales_df.store_id.nunique()}"
axes[0, 0].text(0.05, 0.5, kpi_text, fontsize=13, va="center")
axes[0, 0].set_title("KPI Summary")
region_rev = sales_df.groupby("region").revenue.sum().sort_values(ascending=False)
axes[0, 1].bar(region_rev.index, region_rev.values, color="teal")
axes[0, 1].set_title("Revenue by Region")
month_rev = sales_df.groupby("month").revenue.sum()
axes[1, 0].plot(month_rev.index, month_rev.values, color="navy")
axes[1, 0].set_title("Total Revenue by Month")
axes[1, 0].tick_params(axis="x", rotation=45)
def ratio(s):
    med = s.median()
    return s.min() / med if med != 0 else np.nan
ratios = sales_df.groupby(["store_id", "product"]).units_sold.agg(ratio)
n_alerts = int((ratios < 0.25).sum())
axes[1, 1].axis("off")
alert_color = "red" if n_alerts > 0 else "green"
axes[1, 1].add_patch(mpatches.Circle((0.5, 0.5), 0.35, color=alert_color, alpha=0.85))
label = f"ALERT — {n_alerts} issue(s) found" if n_alerts > 0 else "OK"
axes[1, 1].text(0.5, 0.5, label, ha="center", va="center", color="white", fontsize=13, weight="bold")
axes[1, 1].set_title("Anomaly Alert")
fig.suptitle("UrbanCart Regional Manager Dashboard", fontsize=14)
plt.tight_layout(); plt.show()
